# ETL - Telemetria de Jogos Digitais

## Projeto Integrado - Equipe 4 | Empresa - MIMA Games & Design LTDA

### **Integrantes do Grupo**:
- Gabriel da Silva Freitas
- José Carlos Pereira Neto
- Lucas Paulino Gomes
- Luís Miguel Vitor
- Maria Luiza Tavares Procópio
- Thierry Antonello Pengo

In [4]:
import pandas as pd
import numpy as np
import re
import csv
from pathlib import Path
from datetime import datetime
import unicodedata

## ETAPA 1 — EXTRACT

In [ ]:
# Verificar pasta
pasta_entrada = Path("/opt/projeto/csvs")

if not pasta_entrada.is_dir():
    raise FileNotFoundError(f"A pasta não existe: {pasta_entrada}")

# Função para achar o CSV mais recente
def encontrar_csv(pasta, padrao):
    arquivos = [f for f in pasta.iterdir() if padrao.match(f.name)]

    if len(arquivos) == 0:
        raise FileNotFoundError(
            f"Nenhum arquivo encontrado com o padrão '{padrao.pattern}' em: {pasta}"
        )

    # Ordena pelo nome
    return sorted(arquivos, key=lambda f: f.name, reverse=True)[0]

# Padrões e importação
padroes = {
    "compras":       re.compile(r"^[0-9]*-compras_[0-9]{8}_.*\.csv$"),
    "itens":         re.compile(r"^[0-9]*-itens_[0-9]{8}_.*\.csv$"),
    "itens_compras": re.compile(r"^[0-9]*-itens_compra_[0-9]{8}_.*\.csv$"),
    "jogadores":     re.compile(r"^[0-9]*-jogadores_[0-9]+_[0-9]{8}_.*\.csv$"),
    "partidas":      re.compile(r"^[0-9]*-partidas_[0-9]{8}_.*\.csv$"),
    "sessoes":       re.compile(r"^[0-9]*-sessoes_[0-9]{8}_.*\.csv$"),
}

dados = {}
for nome, padrao in padroes.items():
    caminho = encontrar_csv(pasta_entrada, padrao)

    if not caminho.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {caminho}")

    dados[nome] = pd.read_csv(caminho)
    dados[nome]["lote"] = int(padroes[nome].match(caminho.name).group(1) or 0)
    print(f"Importado: {caminho.name} -> {nome}")

compras, itens, itens_compras, jogadores, partidas, sessoes = (dados[k] for k in padroes)

## ETAPA 2 — TRANSFORM

In [ ]:
# Padronização de valores ausentes
def limpa_nulos(df):
    return df.replace(["", "NA", "NULL"], np.nan)

# Remoção de registros duplicados
def remove_duplicados(df):
    return df.drop_duplicates()

dados_nulos   = {nome: limpa_nulos(df) for nome, df in dados.items()}
dados_limpos  = {nome: remove_duplicados(df) for nome, df in dados_nulos.items()}

compras_limpas, itens_limpos, itens_compras_limpos, jogadores_limpos, partidas_limpas, sessoes_limpas = (dados_limpos[k] for k in padroes)

def converter_numero(x):
    s = (
        pd.Series(x)
        .astype("string")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .str.replace(",", ".", n=1, regex=False)
    )
    return pd.to_numeric(s, errors="coerce")   # as.numeric (inválido vira NaN)

def chave_texto(x):
    x = pd.Series(x).astype("string")

    x = x.str.replace(r"\s+", " ", regex=True).str.strip()

    x = x.map(
        lambda texto: unicodedata.normalize("NFKD", texto)
            .encode("ASCII", "ignore")
            .decode("ASCII")
            .lower()
        if pd.notna(texto) else pd.NA
    )

    return x

def padronizar_categoria(x, validos, nome="coluna"):
    x = pd.Series(x).astype("string")

    original = x.str.replace(r"\s+", " ", regex=True).str.strip()

    chaves = chave_texto(original)
    chaves_validas = chave_texto(validos)

    # Cria um dicionário de correspondência, preservando a primeira ocorrência.
    mapa = {}

    for chave, valor in zip(chaves_validas, validos):
        if pd.notna(chave) and chave not in mapa:
            mapa[chave] = valor

    # Substitui pelos valores padronizados.
    resultado = chaves.map(mapa).astype("string")
    resultado = resultado.where(resultado.notna(), original)

    # Identifica valores que não pertencem à lista.
    fora_da_lista = resultado[
        chaves.notna() & ~chaves.isin(mapa)
    ].drop_duplicates()

    if not fora_da_lista.empty:
        print(
            f"Atenção - {nome} - valores fora da lista: "
            f"{', '.join(fora_da_lista.tolist())}"
        )

    return resultado

# Preço -> raridade (preenche apenas quando o preço é inequívoco)
precos_raridade = {
    4.90:  "Comum",
    14.90: "Incomum",
    24.90: "Rara",
    39.90: "Épica",
    59.90: "Lendária",
    79.90: "Lendária",
}

def raridade_por_preco(preco):
    condicoes = [np.isclose(preco, p, rtol=0, atol=1.5e-8) for p in precos_raridade]
    return pd.Series(
        np.select(condicoes, list(precos_raridade.values()), default=None),
        index=preco.index,
        dtype="object",
    )

itens_limpos["preco_base"] = pd.to_numeric(itens_limpos["preco_base"], errors="coerce")

itens_limpos["raridade"] = itens_limpos["raridade"].fillna(
    raridade_por_preco(itens_limpos["preco_base"])
)

# Marcando descontos inválidos
colunas_num = ["valor_unitario", "desconto", "valor_total_item"]

itens_compras_limpos[colunas_num] = itens_compras_limpos[colunas_num].apply(converter_numero)

# desconto_invalido (recalculado após a conversão)
d = itens_compras_limpos["desconto"]
itens_compras_limpos["desconto_invalido"] = (
    ((d < 0) | (d > 1)).where(d.notna()).astype("boolean")
)

# Correção de datas
def datas_corrigidas(x, hora=False, tz="UTC"):
    x = (
        pd.Series(x, dtype="string")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )
    sufixo = " %H:%M:%S" if hora else ""

    def ler(padrao, formato, trocar_hifen=False):
        mascara = x.str.contains(padrao, regex=True, na=False)
        txt = x.where(mascara)

        if trocar_hifen:
            txt = txt.str.replace("-", "/", regex=False)

        # O que não casa com o formato vira NaT
        out = pd.to_datetime(txt, format=formato + sufixo, errors="coerce")

        if hora:
            return out.dt.tz_localize(tz)
        return out.dt.normalize()

    return {
        "iso": ler(r"^[0-9]{4}-[0-9]{2}-[0-9]{2}", "%Y-%m-%d"),
        "dmy": ler(r"^[0-9]{2}[/-][0-9]{2}[/-][0-9]{4}", "%d/%m/%Y", trocar_hifen=True),
        "mdy": ler(r"^[0-9]{2}/[0-9]{2}/[0-9]{4}", "%m/%d/%Y"),
    }


# Garante que as colunas ainda são texto
def exigir_texto(df, colunas):
    for col in colunas:
        if not pd.api.types.is_string_dtype(df[col]):
            raise TypeError(
                f"A coluna '{col}' já foi convertida. "
                "Rode esta etapa a partir do objeto com as datas em texto."
            )


# Segundos desde 01/01/1970 -> data e hora
def segundos_para_posix(s, tz="UTC"):
    return pd.to_datetime(pd.Series(s), unit="s", utc=True).dt.tz_convert(tz)


# Ambígua: DMY e MDY válidas e diferentes
def eh_ambigua(cand):
    return (
        cand["iso"].isna()
        & cand["dmy"].notna()
        & cand["mdy"].notna()
        & (cand["dmy"] != cand["mdy"])
    )

exigir_texto(compras_limpas, ["data_compra"])
cand = datas_corrigidas(compras_limpas["data_compra"])
compras_limpas["ambigua"] = eh_ambigua(cand)

import numpy as np
import pandas as pd

def resolve_por_duracao(ini_txt, fim_txt, duracao, tz="UTC"):
    ci = datas_corrigidas(ini_txt, hora=True, tz=tz)
    cf = datas_corrigidas(fim_txt, hora=True, tz=tz)
    idx = ci["iso"].index

    origem = pd.Timestamp("1970-01-01", tz=tz)

    def em_segundos(s):
        return (s - origem).dt.total_seconds().to_numpy(dtype=float)

    # Coluna 0: ISO; coluna 1: DMY; coluna 2: MDY
    ini_num = np.column_stack([em_segundos(ci[k]) for k in ("iso", "dmy", "mdy")])
    fim_num = np.column_stack([em_segundos(cf[k]) for k in ("iso", "dmy", "mdy")])

    duracao = pd.to_numeric(pd.Series(duracao), errors="coerce").to_numpy(dtype=float)

    combos = [(i, f) for f in range(3) for i in range(3)]
    i_idx = np.array([i for i, _ in combos])
    f_idx = np.array([f for _, f in combos])

    # Erro de cada combinação em relação à duração informada (n x 9)
    erros = np.abs((fim_num[:, f_idx] - ini_num[:, i_idx]) / 60 - duracao[:, None])

    # Combinações com valores ausentes não podem ser escolhidas
    erros = np.where(np.isnan(erros), np.inf, erros)

    # Combinação de menor erro em cada linha (empate: a primeira)
    melhor = erros.argmin(axis=1)
    pos = np.arange(len(melhor))
    erro = erros[pos, melhor]

    ini = ini_num[pos, i_idx[melhor]]
    fim = fim_num[pos, f_idx[melhor]]

    # Sem combinação validável: prioriza ISO, depois DMY e por último MDY
    def coalesce_colunas(m):
        out = m[:, 0].copy()
        for j in (1, 2):
            out = np.where(np.isnan(out), m[:, j], out)
        return out

    sem_ref = np.isinf(erro)
    ini = np.where(sem_ref, coalesce_colunas(ini_num), ini)
    fim = np.where(sem_ref, coalesce_colunas(fim_num), fim)

    return {
        "ini": segundos_para_posix(pd.Series(ini, index=idx), tz),
        "fim": segundos_para_posix(pd.Series(fim, index=idx), tz),
        "erro": pd.Series(erro, index=idx),
        "ambigua": eh_ambigua(ci) | eh_ambigua(cf),
    }

# Sessões e partidas:
def corrigir_datas_por_duracao(df, col_ini, col_fim, col_duracao="duracao_minutos"):
    exigir_texto(df, [col_ini, col_fim])

    r = resolve_por_duracao(df[col_ini], df[col_fim], df[col_duracao])

    return df.assign(**{col_ini: r["ini"], col_fim: r["fim"]})

sessoes_corrigidas = corrigir_datas_por_duracao(
    sessoes_limpas, "data_inicio_sessao", "data_fim_sessao"
)
partidas_corrigidas = corrigir_datas_por_duracao(
    partidas_limpas, "data_inicio_partida", "data_fim_partida"
)


# Compras
# Requer que `sessoes` já tenha as datas corrigidas.
def corrigir_datas_compras(compras, sessoes):
    exigir_texto(compras, "data_compra") if False else exigir_texto(compras, ["data_compra"])

    cand = datas_corrigidas(compras["data_compra"], hora=True)
    iso, dmy, mdy = cand["iso"], cand["dmy"], cand["mdy"]

    # Sessão correspondente a cada compra (match = primeira ocorrência)
    s = sessoes.drop_duplicates("sessao_id").set_index("sessao_id")
    ini = compras["sessao_id"].map(s["data_inicio_sessao"])
    fim = compras["sessao_id"].map(s["data_fim_sessao"])

    def dentro(x):
        return x.notna() & ini.notna() & (x >= ini) & (x <= fim)

    # Prioridade: ISO > DMY dentro > MDY dentro > DMY > MDY
    nova = (
        iso
        .combine_first(dmy.where(dentro(dmy)))
        .combine_first(mdy.where(dentro(mdy)))
        .combine_first(dmy)
        .combine_first(mdy)
    )

    return compras.assign(data_compra=nova)

compras_corrigidas = corrigir_datas_compras(compras_limpas, sessoes_corrigidas)


# Jogadores
# Idade completa em uma data de referência
def idade_em(nasc, ref):
    nasc = pd.Series(pd.to_datetime(nasc))
    ref = pd.Series(pd.to_datetime(ref), index=nasc.index)  # aceita escalar ou Series

    anos = ref.dt.year - nasc.dt.year
    antes_do_aniversario = (nasc.dt.month * 100 + nasc.dt.day) > (ref.dt.month * 100 + ref.dt.day)
    return anos - antes_do_aniversario.astype(int)

EPOCH = pd.Timestamp("1970-01-01")

def em_dias(s):
    return (s - EPOCH).dt.days.to_numpy(dtype=float)

def coalesce_colunas(m):
    out = m[:, 0].copy()
    for j in (1, 2):
        out = np.where(np.isnan(out), m[:, j], out)
    return out

# Requer que `sessoes` já tenha as datas corrigidas.
def corrigir_datas_jogadores(jogadores, sessoes, data_ref=pd.Timestamp("2026-10-19")):
    exigir_texto(jogadores, ["data_nascimento", "data_criacao_conta"])

    n = len(jogadores)
    pos = np.arange(n)
    idx = jogadores.index

    # ---- PARTE 1: data de nascimento ----
    cn = datas_corrigidas(jogadores["data_nascimento"], hora=False)
    chaves = ("iso", "dmy", "mdy")

    num_n = np.column_stack([em_dias(cn[k]) for k in chaves])

    # Diferença entre a idade de cada interpretação e a idade informada (n x 3)
    idade = pd.to_numeric(jogadores["idade"], errors="coerce")
    erros_n = np.column_stack([
        np.abs(idade_em(cn[k], data_ref) - idade).to_numpy(dtype=float)
        for k in chaves
    ])
    erros_n = np.where(np.isnan(erros_n), np.inf, erros_n)

    melhor_n = erros_n.argmin(axis=1)
    nasc_num = num_n[pos, melhor_n]

    # Sem referência válida: prioriza ISO, DMY e MDY
    sem_ref = np.isinf(erros_n[pos, melhor_n])
    nasc_num = np.where(sem_ref, coalesce_colunas(num_n), nasc_num)

    nasc = pd.Series(pd.to_datetime(nasc_num, unit="D"), index=idx)

    # ---- PARTE 2: data de criação da conta ----
    cc = datas_corrigidas(jogadores["data_criacao_conta"], hora=False)

    c_iso, c_dmy, c_mdy = (em_dias(cc[k]) for k in chaves)

    # Limite inferior: 13 anos de idade e 2010-01-01
    dias_2010 = (pd.Timestamp("2010-01-01") - EPOCH).days
    lim_inf = np.fmax(em_dias(nasc + pd.DateOffset(years=13)), dias_2010)

    # Limite superior: primeira sessão do jogador, e nunca depois de data_ref
    ini_sessao = (
        sessoes["data_inicio_sessao"]
        .dt.tz_convert("UTC").dt.tz_localize(None)   # as.Date() do POSIXct em UTC
        .dt.normalize()
    )
    prim = pd.Series(em_dias(ini_sessao), index=sessoes.index).groupby(sessoes["player_id"]).min()

    lim_sup = jogadores["player_id"].map(prim).to_numpy(dtype=float)
    lim_sup = np.fmin(lim_sup, (data_ref - EPOCH).days)

    def dentro(x):
        return ~np.isnan(x) & (x >= lim_inf) & (x <= lim_sup)

    # Prioridade: ISO > DMY dentro dos limites > MDY dentro dos limites > DMY > MDY
    cria = np.where(~np.isnan(c_iso), c_iso,
           np.where(dentro(c_dmy), c_dmy,
           np.where(dentro(c_mdy), c_mdy,
           np.where(~np.isnan(c_dmy), c_dmy, c_mdy))))

    return jogadores.assign(
        data_nascimento=nasc,
        data_criacao_conta=pd.Series(pd.to_datetime(cria, unit="D"), index=idx),
    )

jogadores_corrigidos = corrigir_datas_jogadores(jogadores_limpos, sessoes_corrigidas)

## ETAPA 3 — LOAD

In [ ]:
pasta_saida = Path("/opt/projeto/csvs_tratados")
pasta_saida.mkdir(parents=True, exist_ok=True)

saidas = {
    "compras":       compras_corrigidas,
    "itens":         itens_limpos,
    "itens_compras": itens_compras_limpos,
    "jogadores":     jogadores_corrigidos,
    "partidas":      partidas_corrigidas,
    "sessoes":       sessoes_corrigidas,
}

def extrair_lote(caminhos):
    if isinstance(caminhos, (str, Path)):
        caminhos = [caminhos]
    lotes = sorted({re.sub(r"^([0-9]+)-.*$", r"\1", Path(c).name) for c in caminhos})
    if len(lotes) != 1:
        raise ValueError(f"Esperava 1 lote, mas foi encontrado: {', '.join(lotes)}")
    return lotes[0]

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

for nome, df in saidas.items():
    lote = extrair_lote(caminho[nome])
    destino = pasta_saida / f"{lote}-{nome}_{timestamp}_limpo.csv"
    df.to_csv(destino, index=False, encoding="utf-8", na_rep="NA", quoting=csv.QUOTE_NONNUMERIC)
    print(f"Exportado: {destino}")